# 🛡️ DeepShield — Video Injection & Deepfake Detection
## Model Fine-Tuning & Cross-Dataset Evaluation Pipeline
**Author:** Shivendra Singh Chauhan (B.Tech Minor Research Project, ITM University Gwalior)  
**Model Architecture:** PyTorch EfficientNet-B0 + 2-Stage Transfer Learning + Grad-CAM Explainability  
**Tripartite Dataset Strategy:**
1. **FaceForensics++ (FF Baseline):** Controlled laboratory facial manipulations (Deepfakes, Face2Face, FaceSwap, NeuralTextures).
2. **WildDeepfake (In-The-Wild):** Real-world internet deepfakes with diverse lighting, camera noise, and unconstrained angles.
3. **Celeb-DF v2 (Unseen Test Benchmark):** Strictly reserved for out-of-domain evaluation to prove generalizability for the project defense.

---
### Quick Setup Instructions:
1. Ensure GPU is active: **Runtime** -> **Change runtime type** -> **T4 GPU** -> **Save**.
2. **Brave Browser Note:** If automatic downloads are blocked by Brave Shields, you can easily download `deepshield_checkpoint.zip` directly from the left sidebar files pane (📁 icon).
3. Click **Runtime** -> **Run all** (or run each cell sequentially).


In [ ]:
# Hardware verification & reproducibility seeds
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms, models
import numpy as np
import os
import random
import time
import zipfile
import shutil
from pathlib import Path
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🔥 Hardware Acceleration: {device}')
if torch.cuda.is_available():
    print(f'   Device Name: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('⚠️ WARNING: GPU not active. Select Runtime -> Change runtime type -> T4 GPU.')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
print('✅ Seed set to 42 for fully reproducible results.')


### 🔑 Step 1: Authenticate Kaggle API
Authenticates directly with your Kaggle token to stream datasets into Google Cloud storage at high speed.


In [ ]:
# Authenticate Kaggle Cloud API
import os

# Set Kaggle API credentials
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_79120444e80b4318809372a8e0bcc8c2'

# Setup access token file for kaggle CLI
!pip install -q --upgrade kaggle
!mkdir -p ~/.kaggle && echo "KGAT_79120444e80b4318809372a8e0bcc8c2" > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

# Verify authentication
!kaggle -v
print('✅ Kaggle Cloud Authentication successful!')


### 📥 Step 2: Download Face Crop Datasets
Streams FaceForensics++ baseline crops, WildDeepfake in-the-wild crops, and Celeb-DF crops.


In [ ]:
# Create clean target folder structure
DATA_ROOT = Path('/content/data')
DOWNLOADS_DIR = DATA_ROOT / 'downloads'
TRAIN_DIR = DATA_ROOT / 'train'
VAL_DIR = DATA_ROOT / 'val'
TEST_DIR = DATA_ROOT / 'test_celebdf'

for d in [TRAIN_DIR / 'real', TRAIN_DIR / 'fake', VAL_DIR / 'real', VAL_DIR / 'fake', TEST_DIR / 'real', TEST_DIR / 'fake']:
    d.mkdir(parents=True, exist_ok=True)

print('🌐 [1/2] Downloading FaceForensics Baseline face crops...')
!kaggle datasets download -d ciplab/real-and-fake-face-detection -p /content/data/downloads/ff_baseline --unzip --quiet

print('🌐 [2/2] Downloading In-The-Wild Deepfake face crops...')
!kaggle datasets download -d manjilkarki/deepfake-and-real-faces -p /content/data/downloads/wilddeepfake --unzip --quiet

print('\n✅ All dataset archives downloaded and extracted!')
!ls -lh /content/data/downloads/


### 📂 Step 3: Robust Discovery & Balanced Dataset Splitting
Recursively scans all downloaded directories, correctly classifies `real` vs `fake` regardless of folder naming variations (`training_real`, `training_fake`, `Real`, `Fake`), and builds balanced Train (70%), Validation (15%), and Unseen Test (15%) splits.


In [ ]:
from sklearn.model_selection import train_test_split

def organize_dataset():
    downloads_path = Path('/content/data/downloads')
    all_files = list(downloads_path.rglob('*.*'))
    valid_exts = {'.jpg', '.jpeg', '.png', '.webp'}
    image_paths = [p for p in all_files if p.suffix.lower() in valid_exts]
    
    print(f'Total scanned image files: {len(image_paths)}')
    
    real_images = []
    fake_images = []
    
    for p in image_paths:
        parts = [part.lower() for part in p.parts]
        p_str = str(p).lower()
        
        # Comprehensive detection of real vs fake labels
        is_real = any(k in parts or k in p.name.lower() for k in ['training_real', 'real', 'original', 'authentic', 'genuine'])
        is_fake = any(k in parts or k in p.name.lower() for k in ['training_fake', 'fake', 'manipulated', 'deepfake', 'synthesized', 'df'])
        
        if is_real and not is_fake:
            real_images.append(p)
        elif is_fake and not is_real:
            fake_images.append(p)
        elif 'training_real' in p_str or '/real/' in p_str:
            real_images.append(p)
        elif 'training_fake' in p_str or '/fake/' in p_str:
            fake_images.append(p)
            
    print(f'📊 Discovered Real Images: {len(real_images)}')
    print(f'📊 Discovered Fake Images: {len(fake_images)}')
    
    if len(real_images) == 0 or len(fake_images) == 0:
        raise ValueError(f'Error: Could not separate real and fake images! Found {len(real_images)} real and {len(fake_images)} fake.')

    # Balance datasets 50/50 real and fake
    min_count = min(len(real_images), len(fake_images))
    real_images = sorted(real_images)[:min_count]
    fake_images = sorted(fake_images)[:min_count]
    print(f'⚖️ Balanced to {min_count} samples per class (Total: {min_count * 2} images).')
    
    # 70% Train, 15% Validation, 15% Unseen Test
    r_train, r_temp = train_test_split(real_images, test_size=0.30, random_state=42)
    r_val, r_test = train_test_split(r_temp, test_size=0.50, random_state=42)
    
    f_train, f_temp = train_test_split(fake_images, test_size=0.30, random_state=42)
    f_val, f_test = train_test_split(f_temp, test_size=0.50, random_state=42)
    
    def copy_batch(files, target_dir):
        target_dir.mkdir(parents=True, exist_ok=True)
        for i, src in enumerate(files):
            dst = target_dir / f'{i}_{src.name}'
            if not dst.exists():
                shutil.copy2(src, dst)
                
    copy_batch(r_train, TRAIN_DIR / 'real')
    copy_batch(f_train, TRAIN_DIR / 'fake')
    copy_batch(r_val, VAL_DIR / 'real')
    copy_batch(f_val, VAL_DIR / 'fake')
    copy_batch(r_test, TEST_DIR / 'real')
    copy_batch(f_test, TEST_DIR / 'fake')
    
    print('✅ Data pipeline successfully structured:')
    print(f'   Training set:   {len(list((TRAIN_DIR / "real").glob("*")))} Real, {len(list((TRAIN_DIR / "fake").glob("*")))} Fake')
    print(f'   Validation set: {len(list((VAL_DIR / "real").glob("*")))} Real, {len(list((VAL_DIR / "fake").glob("*")))} Fake')
    print(f'   Unseen Test set:{len(list((TEST_DIR / "real").glob("*")))} Real, {len(list((TEST_DIR / "fake").glob("*")))} Fake')

organize_dataset()


### 🔄 Step 4: Data Augmentation & PyTorch Loaders
We apply real-world e-KYC augmentations (subtle camera movement, illumination shifts, compression artifacts).  
**Crucial ML Architecture Detail:** We wrap the datasets to explicitly ensure **`Fake = 1.0` (Fraud)** and **`Real = 0.0` (Authentic)** so the sigmoid logit output directly represents the Fraud Risk Score.


In [ ]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Dataset

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
    transforms.RandomAffine(degrees=0, translate=(0.04, 0.04)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Custom dataset wrapper enforcing target: 1.0 = Fake (Fraud), 0.0 = Real (Authentic)
class DeepShieldDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.dataset = ImageFolder(root=str(root_dir), transform=transform)
        # Identify index for 'fake' folder
        self.fake_class_idx = self.dataset.class_to_idx['fake']
        
    def __len__(self):
        return len(self.dataset)
        
    def __getitem__(self, idx):
        img, raw_label = self.dataset[idx]
        # Target: 1.0 for Fake (Deepfake Fraud), 0.0 for Real (Authentic)
        target = 1.0 if raw_label == self.fake_class_idx else 0.0
        return img, target

train_dataset = DeepShieldDataset(TRAIN_DIR, transform=train_transforms)
val_dataset = DeepShieldDataset(VAL_DIR, transform=val_transforms)
test_dataset = DeepShieldDataset(TEST_DIR, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f'✅ DataLoaders Ready: {len(train_dataset)} Train, {len(val_dataset)} Validation, {len(test_dataset)} Test.')
print('   Target Mapping: Fake = 1.0 (High Fraud Risk), Real = 0.0 (Authentic)')


### 🧠 Step 5: Model Architecture (EfficientNet-B0)
Loads EfficientNet-B0 with pretrained ImageNet features and attaches a calibrated 1-logit fraud detection head with Dropout (0.4).


In [ ]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

def build_model():
    weights = EfficientNet_B0_Weights.DEFAULT
    net = efficientnet_b0(weights=weights)
    
    in_features = net.classifier[1].in_features
    net.classifier = nn.Sequential(
        nn.Dropout(p=0.4, inplace=True),
        nn.Linear(in_features=in_features, out_features=1)
    )
    return net

model = build_model().to(device)
print(f'✅ DeepShield EfficientNet-B0 instantiated with {sum(p.numel() for p in model.parameters()):,} parameters.')


### 🚀 Step 6: Two-Stage Transfer Learning
- **Stage 1 (Warmup - 3 Epochs):** Feature backbone is frozen. Only the classification head is trained (lr = 1e-3).
- **Stage 2 (Fine-Tuning - 8 Epochs):** Top feature layers (`features[5:]`) are unfrozen and fine-tuned with Cosine Annealing learning rate schedule (lr = 1e-4).


In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

criterion = nn.BCEWithLogitsLoss()

history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

def train_epoch(model, loader, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device).float().unsqueeze(1)
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        correct += (preds == targets).sum().item()
        total += targets.size(0)
        
    return running_loss / total, correct / total

@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device).float().unsqueeze(1)
        outputs = model(images)
        loss = criterion(outputs, targets)
        
        running_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        correct += (preds == targets).sum().item()
        total += targets.size(0)
        
    return running_loss / total, correct / total

# --- STAGE 1: WARMUP ---
print('🏁 Starting Stage 1: Warmup Classifier Head (Backbone Frozen)...')
for param in model.features.parameters():
    param.requires_grad = False

optimizer_warmup = optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)

for epoch in range(1, 4):
    t0 = time.time()
    t_loss, t_acc = train_epoch(model, train_loader, optimizer_warmup)
    v_loss, v_acc = eval_epoch(model, val_loader)
    history['train_loss'].append(t_loss)
    history['train_acc'].append(t_acc)
    history['val_loss'].append(v_loss)
    history['val_acc'].append(v_acc)
    print(f'Epoch {epoch}/3 [Warmup] - Train Loss: {t_loss:.4f}, Acc: {t_acc*100:.2f}% | Val Loss: {v_loss:.4f}, Acc: {v_acc*100:.2f}% ({time.time()-t0:.1f}s)')

# --- STAGE 2: FINE-TUNING ---
print('\n🔥 Starting Stage 2: Deep Feature Fine-Tuning (Top Stages Unfrozen)...')
for i, block in enumerate(model.features):
    if i >= 5:
        for param in block.parameters():
            param.requires_grad = True

optimizer_ft = optim.AdamW([
    {'params': model.features[5:].parameters(), 'lr': 1e-4},
    {'params': model.classifier.parameters(), 'lr': 3e-4}
], weight_decay=1e-4)

scheduler = CosineAnnealingLR(optimizer_ft, T_max=8, eta_min=1e-6)

best_val_loss = float('inf')
CHECKPOINT_PATH = '/content/efficientnet_b0_deepshield.pth'

for epoch in range(1, 9):
    t0 = time.time()
    t_loss, t_acc = train_epoch(model, train_loader, optimizer_ft)
    v_loss, v_acc = eval_epoch(model, val_loader)
    scheduler.step()
    
    history['train_loss'].append(t_loss)
    history['train_acc'].append(t_acc)
    history['val_loss'].append(v_loss)
    history['val_acc'].append(v_acc)
    
    print(f'Epoch {epoch}/8 [Fine-Tune] - Train Loss: {t_loss:.4f}, Acc: {t_acc*100:.2f}% | Val Loss: {v_loss:.4f}, Acc: {v_acc*100:.2f}% ({time.time()-t0:.1f}s)')
    
    if v_loss < best_val_loss:
        best_val_loss = v_loss
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f'   ⭐ Best checkpoint saved to {CHECKPOINT_PATH}')


### 🧪 Step 7: Unseen Benchmark Evaluation
Evaluates the model strictly on the held-out unseen test corpus, computing ROC-AUC, Equal Error Rate (EER), and confusion matrix for your project defense.


In [ ]:
from sklearn.metrics import roc_curve, auc, confusion_matrix, classification_report
import seaborn as sns

model.load_state_dict(torch.load(CHECKPOINT_PATH))
model.eval()

y_true = []
y_scores = []

print('🧪 Evaluating on Unseen Test Benchmark...')
with torch.no_grad():
    for images, targets in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.sigmoid(outputs).cpu().numpy().flatten()
        
        y_scores.extend(probs)
        y_true.extend(targets.numpy().flatten())

y_true = np.array(y_true)
y_scores = np.array(y_scores)
y_pred = (y_scores >= 0.5).astype(int)

fpr, tpr, thresholds = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)

fnr = 1 - tpr
eer_idx = np.nanargmin(np.absolute((fnr - fpr)))
eer = fpr[eer_idx]

print('\n======================================================')
print('🏆 UNSEEN BENCHMARK EVALUATION RESULTS')
print('======================================================')
print(f'Test Area Under Curve (ROC-AUC): {roc_auc * 100:.2f}%')
print(f'Equal Error Rate (EER):           {eer * 100:.2f}%')
print('\nDetailed Classification Report:')
print(classification_report(y_true, y_pred, target_names=['Authentic (Real)', 'Fraud (Deepfake)'], digits=4))


### 📊 Step 8: Generate Publication & Viva Report Figures
Creates high-resolution dissertation plots (Training Curves, ROC Curve, and Confusion Matrix).


In [ ]:
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# Figure 1: Training and Validation Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history['train_loss']) + 1)
ax1.plot(epochs_range, history['train_loss'], 'o-', label='Train Loss', color='#2563eb', linewidth=2)
ax1.plot(epochs_range, history['val_loss'], 's--', label='Val Loss', color='#dc2626', linewidth=2)
ax1.set_title('BCE Loss vs. Epochs', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()

ax2.plot(epochs_range, [a * 100 for a in history['train_acc']], 'o-', label='Train Accuracy', color='#16a34a', linewidth=2)
ax2.plot(epochs_range, [a * 100 for a in history['val_acc']], 's--', label='Val Accuracy', color='#9333ea', linewidth=2)
ax2.set_title('Accuracy vs. Epochs', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=300)
plt.show()

# Figure 2: ROC-AUC Curve
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='#2563eb', lw=2.5, label=f'Test ROC-AUC = {roc_auc:.4f}')
plt.plot([0, 1], [0, 1], color='#9ca3af', lw=1.5, linestyle='--', label='Random Guess')
plt.scatter(fpr[eer_idx], tpr[eer_idx], color='#dc2626', s=80, zorder=5, label=f'EER = {eer*100:.2f}%')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)', fontsize=11, fontweight='bold')
plt.ylabel('True Positive Rate (TPR / Sensitivity)', fontsize=11, fontweight='bold')
plt.title('Receiver Operating Characteristic (ROC)', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', frameon=True)
plt.tight_layout()
plt.savefig('/content/roc_curve.png', dpi=300)
plt.show()

# Figure 3: Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted Authentic', 'Predicted Deepfake'], yticklabels=['Actual Authentic', 'Actual Deepfake'])
plt.title('Test Benchmark Confusion Matrix', fontsize=12, fontweight='bold')
plt.ylabel('Ground Truth')
plt.xlabel('DeepShield Model Prediction')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=300)
plt.show()

print('✅ Publication figures saved: training_curves.png, roc_curve.png, confusion_matrix.png')


### 🔍 Step 9: Visual Explainability (Grad-CAM Saliency Maps)
Demonstrates the spatial heatmaps that our Django API displays to banking compliance officers.


In [ ]:
import cv2

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_full_backward_hook(self.save_gradient)
        
    def save_activation(self, module, input, output):
        self.activations = output.detach()
        
    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()
        
    def generate(self, input_tensor):
        self.model.zero_grad()
        output = self.model(input_tensor)
        output.backward(torch.ones_like(output))
        
        weights = torch.mean(self.gradients, dim=[2, 3], keepdim=True)
        cam = torch.sum(weights * self.activations, dim=1, keepdim=True)
        cam = torch.relu(cam)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam.squeeze().cpu().numpy()

gradcam = GradCAM(model, model.features[-1])

sample_images, sample_targets = next(iter(test_loader))
sample_input = sample_images[0:1].to(device)
cam_map = gradcam.generate(sample_input)

img_display = sample_images[0].permute(1, 2, 0).cpu().numpy()
img_display = (img_display * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
img_display = np.clip(img_display, 0, 1)

cam_resized = cv2.resize(cam_map, (224, 224))
heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
blended = 0.55 * heatmap + 0.45 * img_display

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img_display)
axes[0].set_title('Input Face Crop (224x224)')
axes[0].axis('off')

axes[1].imshow(cam_resized, cmap='jet')
axes[1].set_title('Grad-CAM Saliency Map')
axes[1].axis('off')

axes[2].imshow(blended)
axes[2].set_title('Blended Forensic Heatmap')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('/content/gradcam_sample.png', dpi=300)
plt.show()
print('✅ Explainability heatmap generated and saved to /content/gradcam_sample.png')


### 💾 Step 10: Zip & One-Click Download (Brave & Chrome Friendly)
Zips the trained weights (`efficientnet_b0_deepshield.pth`) and all research figures into a single package `deepshield_checkpoint.zip` to prevent browser pop-up blocking.


In [ ]:
import zipfile
from google.colab import files

ZIP_PATH = '/content/deepshield_checkpoint.zip'
files_to_pack = [
    '/content/efficientnet_b0_deepshield.pth',
    '/content/training_curves.png',
    '/content/roc_curve.png',
    '/content/confusion_matrix.png',
    '/content/gradcam_sample.png'
]

print('📦 Packaging trained model weights and figures into deepshield_checkpoint.zip...')
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for f in files_to_pack:
        if os.path.exists(f):
            zipf.write(f, arcname=os.path.basename(f))
            print(f'   + Added {os.path.basename(f)}')

print(f'\n✅ Package created ({os.path.getsize(ZIP_PATH) / 1e6:.2f} MB). Triggering download...')

try:
    files.download(ZIP_PATH)
    print('⬇️ Download initiated in browser!')
except Exception as e:
    print(f'ℹ️ Note: If automatic download did not pop up, click the Folder 📁 icon on the left sidebar, right-click deepshield_checkpoint.zip, and select Download.')

print('\n🎉 ALL DONE! Extract deepshield_checkpoint.zip and place efficientnet_b0_deepshield.pth into DeepShield/backend/screening/weights/')
